# 17 · The static demo page for the blog (`site/`)

Ten example questions are each answered once with the best combination, exactly as the live demo answers them, and
saved; the page in `site/` is the live page in static mode (no server, no language model). Whatever the questions
return is shown, good or bad.

Serve it locally with `uv run python -m http.server 8766 --directory site`.

In [1]:
import json
import os
import time

import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
ROOT = data_root()
DEMO = ROOT / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
meta = json.loads((DEMO / "meta.json").read_text())
LEAVES = cfg["classes"]["leaves"]
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
T_START = time.time()


def build_engine(log_dir):
    """The demo engine as ``askmap.demo.serve`` builds it; language-model replies are saved in ``log_dir``."""

    def ask(question, path):
        return translate(client, llm["models"]["main"], system, question, path, llm)

    planner = make_planner(cfg, ask, log_dir)
    enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
    contexts = make_contexts(cfg, meta, DEMO)
    return Engine(
        load_tiles(DEMO, meta),
        LEAVES,
        enc,
        planner,
        contexts,
        meta["best"],
        CAP,
        top_share=meta["top_share"],
        not_ready=meta["not_ready"],
    )

In [2]:
from askmap.demo.site import write_site

QUESTIONS = [
    ("wetlands next to crop fields", "north_dakota"),
    ("corn fields where July 2021 had at least 5 days above 35 C", "north_dakota"),
    ("corn fields where July 2021 had at least 5 days above 35 C", "north_carolina"),
    ("grassland near an existing solar farm", "north_carolina"),
    ("cropland where spring 2022 was one of the wettest 10% since 2001", "north_dakota"),
    ("hay fields within 200 m of a wetland", "minnesota"),
    ("soybean fields more than 300 m from any wetland or open water", "minnesota"),
    ("forest within 500 m of a solar farm", "georgia"),
    ("grassland where summer 2022 rainfall was in the driest 10% of summers since 2001", "north_dakota"),
    ("wetlands within 100 m of developed land", "north_carolina"),
]
engine = build_engine(PROJECT_ROOT / "outputs" / "llm" / "demo")
answers, rows = [], []
for q, area in QUESTIONS:
    r = engine.search(q, area, "best")
    f = r["facts"]
    answers.append({"question": q, "area": area, "result": r})
    top = f["land_of_shown_pixels"][0] if f["land_of_shown_pixels"] else None
    rows.append(
        {
            "question": q,
            "area": area,
            "places": f["shown_pixels"],
            "top official class": f"{top['class']} {top['share']:.0%}" if top else "",
            "plan": f["plan_words"],
            "empty reason": f["empty_reason"] or "",
        }
    )
pd.set_option("display.max_colwidth", 140)
pd.DataFrame(rows)

/Volumes/disco2/github/ask-the-map-demo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1030 17:22:25.383000 38855 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  69%|██████▉   | 406/590 [00:00<00:00, 3859.86it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 3987.46it/s]

,question,area,places,top official class,plan,empty reason
0,wetlands next to crop fields,north_dakota,4053,wetland 40%,"Wetland, next to cropland.",
1,corn fields where July 2021 had at least 5 days above 35 C,north_dakota,19172,corn 93%,"Corn, where July 2021 had at least 5 days with daily high temperature above 35 °C (NASA POWER).",
2,corn fields where July 2021 had at least 5 days above 35 C,north_carolina,0,,"Corn, where July 2021 had at least 5 days with daily high temperature above 35 °C (NASA POWER).","No NASA POWER cell in this area meets ""July 2021 had at least 5 days with daily high temperature above 35 °C"": the closest cell had 2 ag..."
3,grassland near an existing solar farm,north_carolina,578,grassland 46%,"Grassland, within 300 m of solar farm.",
4,cropland where spring 2022 was one of the wettest 10% since 2001,north_dakota,48214,soybeans 51%,"Cropland, where March to May 2022: total rainfall in the highest 10% of years 2001 to 2025 (NASA POWER).",
5,hay fields within 200 m of a wetland,minnesota,165,grassland 40%,"Hay, within 200 m of wetland.",
6,soybean fields more than 300 m from any wetland or open water,minnesota,13768,soybeans 95%,"Soybeans, more than 300 m from any wetland or water.",
7,forest within 500 m of a solar farm,georgia,623,forest 70%,"Forest, within 500 m of solar farm.",
8,grassland where summer 2022 rainfall was in the driest 10% of summers since 2001,north_dakota,21550,grassland 82%,"Grassland, where June to August 2022: total rainfall in the lowest 10% of years 2001 to 2025 (NASA POWER).",
9,wetlands within 100 m of developed land,north_carolina,5347,wetland 62%,"Wetland, within 100 m of developed.",


In [3]:
areas = {k: {"label": a["label"], "tiles": a["tiles"]} for k, a in meta["areas"].items()}
page = (PROJECT_ROOT / "src" / "askmap" / "demo" / "index.html").read_text()
files = write_site(PROJECT_ROOT / "site", page, areas, answers)
mb = sum(p.stat().st_size for p in files) / 1e6
record = {"questions": rows, "files": [str(p.relative_to(PROJECT_ROOT)) for p in files], "MB": round(mb, 1)}
(RES / "17_site.json").write_text(json.dumps(record, indent=1))
print(f"{len(files)} files, {mb:.1f} MB; run time {(time.time() - T_START) / 60:.1f} min")

12 files, 0.2 MB; run time 0.4 min
